In [ ]:
using CairoMakie;       #Paquetería para generar gráficos de los datos
using Colors;           #Paquetería para usar RGB en las gráficas de Makie
using DelimitedFiles;   #Paquetería para leer y escribir archivos .csv
using LaTeXStrings;     #Paquetería para emplear formato LaTeX en las gráficas
using LinearAlgebra;    #Paquetería para emplear funciones básicas de álgebra lineal

#Cargamos las funciones necesarias para generar vecindades cuasiperiódicas
PathFunctions = "Quasiperiodic-Tiles/Global Structural Studies/Functions/";
include(PathFunctions * "Quasicrystals.jl");

#Algunas funciones adicionales requeridas
AproxLambda(x) = (2π/(1 - cos(2π/x))); #Dimensiones de nuestro aproximante estadístico
area(rombo) = abs(0.5*sum((rombo[i][2]+rombo[mod1(i+1,4)][2])*(rombo[i][1]-rombo[mod1(i+1,4)][1]) for i in 1:4)); #Función que calcula el área de un rombo

#Estructura para manejar los lados de cada tesela como una conexión en un grafo
struct Spring
    S1::Vector{BigFloat}
    S2::Vector{BigFloat}
end

#Función que determina si una conexión en un grafo es la misma o no, sin considerar direccionalidad
function is_Equal(Spring1, Spring2)
    if (Spring1.S1 == Spring2.S1) && (Spring1.S2 == Spring2.S2)
        return true
    elseif (Spring1.S1 == Spring2.S2) && (Spring1.S2 == Spring2.S1)
        return true
    else
        return false
    end
end

## Definición del lienzo y su distribución espacial

In [ ]:
#Configuración inicial del lienzo
fig = Figure(size = (1250, 1000), figure_padding = (0, 30, 5, 0)); #Contenedor general donde se graficarán los datos; para el padding (izq, der, abajo, arriba)
LabelFontSize = 35; #Tamaño de las etiquetas X y Y a emplear
TickLabelSize = 33; #Tamaño del grosor de los ticks a emplear

#Fila de la hilera de vecindades locales
Fila_Vecindades = Axis(
                       fig[1, 1],
                       aspect = DataAspect(),
                       backgroundcolor = :transparent
                      );
hidedecorations!(Fila_Vecindades);  #Ocultar los ejes y el mallado de la gráfica
hidespines!(Fila_Vecindades);       #Ocultar bordes

#Approx_Clasico contiene la información del fondo de las teselas más pequeñas del aproximante clásico y la vecindad local de nuestro aproximante
Approx_Clasico = Axis(
                      fig[2:4, 1],
                      aspect = DataAspect(),
                      backgroundcolor = :transparent
                     );
hidedecorations!(Approx_Clasico);   #Ocultar los ejes y el mallado de la gráfica
hidespines!(Approx_Clasico);        #Ocultar bordes

#Local_Origin_N15 contiene las teselas de una vecindad local alrededor del origen de la simetría rotacional N = 15
Local_Origin_N15 = Axis(
                        fig[1, 2],
                        aspect = DataAspect(),
                        backgroundcolor = :transparent
                       );
hidedecorations!(Local_Origin_N15); #Ocultar los ejes y el mallado de la gráfica
hidespines!(Local_Origin_N15);      #Ocultar bordes

#Sigma2_2D contiene la información de la gráfica de hiperuniformidad de los sistemas seleccionados 2D
Sigma2_2D = Axis(
                 fig[2, 2],
                 limits = ((0, 470), nothing),      #Límites de visualización de los ejes X y Y
                 rightspinevisible = false,         #Oculta la línea vertical derecha
                 topspinevisible = false,           #Oculta la línea horizontal superior
                 xgridvisible = false,              #Mallado en los ejes horizontales
                 ygridvisible = false,              #Mallado en los ejes verticales
                 backgroundcolor = :transparent,
                 xticks = [0, 150, 350],            #Posiciones donde colocar los ticks en el eje X
                 yticks = (
                           [0, 2],                  #Posiciones donde colocar los ticks en el eje Y
                           [rich("0"), rich("2")]   #Etiquetas de los ticks en el eje Y
                          ),
                 xlabel = L"R",                     #Etiqueta del eje horizontal
                 ylabel = L"\sigma^{2}(R)/R",       #Etiqueta del eje vertical
                 xlabelsize = LabelFontSize,        #Tamaño de la etiqueta horizontal
                 ylabelsize = LabelFontSize,        #Tamaño de la etiqueta vertical
                 xticklabelsize = TickLabelSize,    #Tamaño de los números del eje horizontal
                 yticklabelsize = TickLabelSize     #Tamaño de los números del eje vertical
                );

#gR_2D contiene la información de la gráfica de g(R) de los sistemas seleccionados 2D
gR_2D = Axis(
             fig[3, 2],
             limits = ((0, 470), (5, 87000)),       #Límites de visualización de los ejes X y Y
             yscale = log10,                        #Escala logarítmica de la gráfica
             rightspinevisible = false,             #Oculta la línea vertical derecha
             topspinevisible = false,               #Oculta la línea horizontal superior
             xgridvisible = false,                  #Mallado en los ejes horizontales
             ygridvisible = false,                  #Mallado en los ejes verticales
             backgroundcolor = :transparent,
             xticks = [0, 150, 350],                #Posiciones donde colocar los ticks en el eje X
             yticks = (
                       [10^1, 10^4],                #Posiciones donde colocar los ticks en el eje Y
                       [rich("10", superscript("1")), rich("10", superscript("4"))] #Etiquetas de los ticks en el eje Y
                      ),
             xlabel = L"R",                         #Etiqueta del eje horizontal
             ylabel = L"g(R)",                      #Etiqueta del eje vertical
             xlabelsize = LabelFontSize,            #Tamaño de la etiqueta horizontal
             ylabelsize = LabelFontSize,            #Tamaño de la etiqueta vertical
             xticklabelsize = TickLabelSize,        #Tamaño de los números del eje horizontal
             yticklabelsize = TickLabelSize         #Tamaño de los números del eje vertical
            );

#λ_N contiene la información de las gráficas de cómo crece Lambda como función de la simetría rotacional N de los casos 2D y 1D
λ_N = Axis(
           fig[4, 2],
           limits = ((4.7, 52), nothing),           #Límites de visualización de los ejes X y Y
           yscale = log10,                          #Escala logarítmica de la gráfica
           xscale = log10,                          #Escala logarítmica de la gráfica
           rightspinevisible = false,               #Oculta la línea vertical derecha
           topspinevisible = false,                 #Oculta la línea horizontal superior
           xgridvisible = false,                    #Mallado en los ejes horizontales
           ygridvisible = false,                    #Mallado en los ejes verticales
           backgroundcolor = :transparent,
           xticks = [5, 10, 25, 50],                #Posiciones donde colocar los ticks en el eje X
           yticks = (
                     [1, 10^6],                     #Posiciones donde colocar los ticks en el eje Y
                     [rich("10", superscript("0")), rich("10", superscript("6"))] #Etiquetas de los ticks en el eje Y
                    ),
           xlabel = L"N",                           #Etiqueta del eje horizontal
           ylabel = L"λ_{N}",                       #Etiqueta del eje vertical
           xlabelsize = LabelFontSize,              #Tamaño de la etiqueta horizontal
           ylabelsize = LabelFontSize,              #Tamaño de la etiqueta vertical
           xticklabelsize = TickLabelSize,          #Tamaño de los números del eje horizontal
           yticklabelsize = TickLabelSize           #Tamaño de los números del eje vertical
          );

#Ajuste del tamaño relativo entre la Fila 1 y la Fila 2
rowsize!(fig.layout, 4, Auto());
rowsize!(fig.layout, 3, Auto());
rowsize!(fig.layout, 2, Auto());
rowsize!(fig.layout, 1, Relative(0.25));

#Ajuste del tamaño relativo entre las columnas
colsize!(fig.layout, 1, Auto())
colsize!(fig.layout, 2, Relative(0.25))

#Eliminar espacio entre columnas y filas
colgap!(fig.layout, 0)
rowgap!(fig.layout, 0)
rowgap!(fig.layout, 2, 0)
rowgap!(fig.layout, 3, 0)

fig

## Construcción de las vecindades locales para las diferentes simetrías rotacionales

In [ ]:
Valor_Alpha_Teselas = 1; #Parámetro global para el alpha de relleno de las teselas secundarias
for NSides in 5:2:15
    Areas_Values = nothing;
    Rho = nothing;
    APoint = nothing;
    Error_Margin = nothing;
    raw_colors = nothing;
    Move_Vector = nothing;
    
    if NSides == 5
        Areas_Values = [0.587785, 0.951057];                        #Arreglo con las áreas, de menor a mayor, de las posibles teselas
        Rho = 1.2328979808609704;                                   #Densidad de sitios en una vecindad cuasiperiódica (Decoración vértices)
        APoint = [-3.5705362153805707e11, -3.037695736138369e11];   #Punto arbitrario en el que se generan las vecindades
        Error_Margin = 3;                                           #Margen de error asociado a los enteros asociados a cada vector estrella del GDM
        MoveVector = [0, 0];                                        #Vector con el que se desplaza el centro de la vecindad circular

        #Definición de Colores Base
        raw_colors = [
                      [204, 41, 54]/255,   #Color de la primer tesela
                      [139, 195, 74]/255   #Color de la segunda tesela
                     ];
    elseif NSides == 7
        Areas_Values = [0.433884, 0.781831, 0.974928];              #Arreglo con las áreas, de menor a mayor, de las posibles teselas
        Rho = 1.2517957581185175;                                   #Densidad de sitios en una vecindad cuasiperiódica (Decoración vértices)
        APoint = [-3.5705362153805707e11, -3.037695736138369e11];   #Punto arbitrario en el que se generan las vecindades
        Error_Margin = 3;                                           #Margen de error asociado a los enteros asociados a cada vector estrella del GDM
        MoveVector = [8, 0];                                        #Vector con el que se desplaza el centro de la vecindad circular
        
        #Definición de Colores Base
        raw_colors = [
                      [208, 0, 0]/255,   #Color de la primer tesela
                      [28, 49, 68]/255,  #Color de la segunda tesela
                      [255, 186, 8]/255  #Color de la tercer tesela
                     ];
    elseif NSides == 9
        Areas_Values = [0.34202, 0.642788, 0.866025, 0.984808];     #Arreglo con las áreas, de menor a mayor, de las posibles teselas
        Rho = 1.260284085456272;                                    #Densidad de sitios en una vecindad cuasiperiódica (Decoración vértices)
        APoint = [-3.5705362153805707e11, -3.037695736138369e11];   #Punto arbitrario en el que se generan las vecindades
        Error_Margin = 3;                                           #Margen de error asociado a los enteros asociados a cada vector estrella del GDM
        MoveVector = [8 + 12, 0];                                   #Vector con el que se desplaza el centro de la vecindad circular

        #Definición de Colores Base
        raw_colors = [
                      [214, 40, 40]/255,  #Color de la primer tesela
                      [0, 48, 73]/255,    #Color de la segunda tesela
                      [247, 127, 0]/255,  #Color de la tercer tesela
                      [252, 191, 73]/255  #Color de la cuarta tesela
                     ];
    elseif NSides == 11
        Areas_Values = [0.281733, 0.540641, 0.75575, 0.909632, 0.989821];   #Arreglo con las áreas, de menor a mayor, de las posibles teselas
        Rho = 1.2645739922427748;                           #Densidad de sitios en una vecindad cuasiperiódica (Decoración vértices)
        APoint = [710645.3206976792, -517631.84949160495];  #Punto correspondiente a la vecindad local que generaremos en otra imagen
        Error_Margin = 3;                                   #Margen de error asociado a los enteros asociados a cada vector estrella del GDM
        MoveVector = [8 + 12 + 18.5, 0];                    #Vector con el que se desplaza el centro de la vecindad circular

        #Definición de Colores Base
        raw_colors = [
                      [214, 40, 40]/255,    #Color de la primer tesela
                      [47, 72, 88]/255,     #Color de la segunda tesela
                      [124, 181, 24]/255,   #Color de la tercer tesela
                      [134, 187, 216]/255,  #Color de la cuarta tesela
                      [242, 100, 25]/255    #Color de la quinta tesela
                     ];
    elseif NSides == 13
        Areas_Values = [0.239316, 0.464723, 0.663123, 0.822984, 0.935016, 0.992709];    #Arreglo con las áreas, de menor a mayor, de las posibles teselas
        Rho = 1.2670366156186388;                                   #Densidad de sitios en una vecindad cuasiperiódica (Decoración vértices)
        APoint = [-3.5705362153805707e11, -3.037695736138369e11];   #Punto arbitrario en el que se generan las vecindades
        Error_Margin = 3;                                           #Margen de error asociado a los enteros asociados a cada vector estrella del GDM
        MoveVector = [8 + 12 + 18.5 + 26, 0];                       #Vector con el que se desplaza el centro de la vecindad circular

        #Definición de Colores Base
        raw_colors = [
                      [205, 0, 26]/255,   #Color de la primer tesela
                      [239, 106, 0]/255,  #Color de la segunda tesela
                      [242, 205, 0]/255,  #Color de la tercer tesela
                      [121, 195, 0]/255,  #Color de la cuarta tesela
                      [25, 97, 174]/255,  #Color de la quinta tesela
                      [97, 0, 125]/255    #Color de la sexta tesela
                     ];
    elseif NSides == 15
        Areas_Values = [0.207912, 0.406737, 0.587785, 0.743145, 0.866025, 0.951057, 0.994522];  #Arreglo con las áreas, de menor a mayor, de las posibles teselas
        Rho = 1.2685820147323164;                                   #Densidad de sitios en una vecindad cuasiperiódica (Decoración vértices)
        APoint = [-3.5705362153805707e11, -3.037695736138369e11];   #Punto arbitrario en el que se generan las vecindades
        Error_Margin = 3;                                           #Margen de error asociado a los enteros asociados a cada vector estrella del GDM
        MoveVector = [8 + 12 + 18.5 + 26 + 35, 0];                  #Vector con el que se desplaza el centro de la vecindad circular

        #Definición de Colores Base
        raw_colors = [
                      [255, 0, 89]/255,     #Color de la primer tesela
                      [255, 140, 0]/255,    #Color de la segunda tesela
                      [180, 230, 0]/255,    #Color de la tercer tesela
                      [15, 255, 219]/255,   #Color de la cuarta tesela
                      [10, 210, 255]/255,   #Color de la quinta tesela
                      [41, 98, 255]/255,    #Color de la sexta tesela
                      [149, 0, 255]/255     #Color de la septima tesela
                     ];
    elseif NSides == 17
        Areas_Values = [0.18375, 0.361242, 0.526432, 0.673696, 0.798017, 0.895163, 0.961826, 0.995734]; #Arreglo con las áreas, de menor a mayor, de las posibles teselas
        Rho = 1.2696141740269873;                                   #Densidad de sitios en una vecindad cuasiperiódica (Decoración vértices)
        APoint = [-3.5705362153805707e11, -3.037695736138369e11];   #Punto arbitrario en el que se generan las vecindades
        Error_Margin = 3;                                           #Margen de error asociado a los enteros asociados a cada vector estrella del GDM
        MoveVector = [8 + 12 + 18.5 + 26 + 35 + 46, 0];             #Vector con el que se desplaza el centro de la vecindad circular

        #Definición de Colores Base
        raw_colors = [
                      [173, 20, 87]/255,    #Color de la primer tesela
                      [244, 67, 54]/255,    #Color de la segunda tesela
                      [255, 152, 0]/255,    #Color de la tercer tesela
                      [255, 193, 7]/255,    #Color de la cuarta tesela
                      [139, 195, 74]/255,   #Color de la quinta tesela
                      [0, 150, 136]/255,    #Color de la sexta tesela
                      [58, 138, 255]/255,   #Color de la septima tesela
                      [21, 101, 192]/255    #Color de la octava tesela
                     ];
    end

    FactNorm = 2*sqrt(π*Rho); #Factor de normalización para las longitudes, genera densidades constantes

    Lambda = AproxLambda(NSides);       #Valor del aproximante estadístico (Tras la normalización de Torquato)
    RadioVecindad = Lambda/FactNorm;    #Valor del radio de la vecindad circular para nuestro aproximante estadístico (Valor previo a la normalización de Torquato)

    Star_Vectors = [[BigFloat(1), 0]];  #Arreglo que contendrá los vectores estrella
    for i in 1:(NSides-1)
        push!(Star_Vectors, [cos((2*i)*pi/NSides), sin((2*i)*pi/NSides)]); #Vértices del polígono regular con NSides
    end
    Alphas_Array = fill(0.0, NSides); #Arreglo con las constantes alfas del GDM
    Average_Distance_Stripes = fill(NSides/2, NSides); #Arreglo con la separación promedio entre las franjas cuasiperiódicas   

    #Construimos los vértices de las teselas del arreglo cuasiperiódico alrededor de APoint
    X_Coord_Site, Y_Coord_Site, Lattice_Sites = quasiperiodic_Neighbourhood(NSides, Error_Margin, RadioVecindad, APoint); #Generamos la vecindad circular
    X_Coord_Site = X_Coord_Site .- APoint[1]; #Recorremos las coordenadas de las teselas alrededor del origen
    Y_Coord_Site = Y_Coord_Site .- APoint[2]; #Recorremos las coordenadas de las teselas alrededor del origen

    Spring_Array = []; #Arreglo que contendrá las conexiones entre vértices
    for i in 1:4:length(X_Coord_Site)
        #Obtenemos las coordenadas de cada tesela en un formato de polígono cerrado (V1-> V2-> V3-> V4-> V1)
        XX = [X_Coord_Site[i], X_Coord_Site[i+1], X_Coord_Site[i+2], X_Coord_Site[i+3], X_Coord_Site[i]]
        YY = [Y_Coord_Site[i], Y_Coord_Site[i+1], Y_Coord_Site[i+2], Y_Coord_Site[i+3], Y_Coord_Site[i]]

        #Conectamos a pares los vértices de las teselas para generar los 4 lados de la misma (V1-V2, V2-V3, V3-V4, V4-V1)
        for j in 1:(length(XX) - 1)
            Tile_Side = Spring([XX[j], YY[j]], [XX[j+1], YY[j+1]]) #Definimos el lado de la tesela que conecta a dos vértices subsecuentes
            Add_Side = true; #Una llave que determina si se guarda la conexión al ser única o se descarta por ser repetida

            for e in Spring_Array #Iteramos sobre todas las conexiones previas
                if is_Equal(e, Tile_Side) #Verificamos si el lado recién creado es una nueva conexión
                    Add_Side = false; #Si la conexión recien creada es igual a una previa, se descarta su adición y se rompe el ciclo
                    break
                end
            end

            if Add_Side == true
                push!(Spring_Array, Tile_Side) #Si tras revisar con todos las conexiones previas no se encuentra un duplicado, entonces se añade la nueva conexión
            end
        end
    end

    palette_rgb = [RGB(c[1], c[2], c[3]) for c in raw_colors]; #Convertimos a objetos RGB para manipularlos fácilmente

    # --- PREPARAR DATOS DE LAS TESELAS (POLÍGONOS) ---
    # Listas vacías para acumular los polígonos y sus colores
    lista_poligonos = Vector{Vector{Point2f}}();
    lista_colores = Vector{RGBA}();

    #Iteramos sobre el índice del primer vértice que conforma cada tesela
    for i in 1:4:length(X_Coord_Site)
        #Extraemos coordenadas (XTile, YTile)
        xs = [X_Coord_Site[k] + MoveVector[1] for k in i:(i+3)];
        ys = [Y_Coord_Site[k] + MoveVector[2] for k in i:(i+3)];
        
        #Calculamos el Área
        XTile_calc = [xs; xs[1]];
        YTile_calc = [ys; ys[1]];
        A = round(Float64(Area(XTile_calc, YTile_calc)), digits = 6);
        
        #Buscar índice del color
        Index = findfirst(x -> x == A, Areas_Values);
        
        #Determinar Alpha y Color final
        alpha_val = (Index == 1) ? 1.0 : Valor_Alpha_Teselas;
        base_c = palette_rgb[Index];
        
        #Guardamos el polígono (como lista de puntos Point2f) y su color RGBA
        push!(lista_poligonos, Point2f.(xs, ys));
        push!(lista_colores, RGBA(base_c.r, base_c.g, base_c.b, alpha_val));
    end

    # --- GRAFICAR TODAS LAS TESELAS DE UNA VEZ ---
    poly!(
          Fila_Vecindades, lista_poligonos, 
          color = lista_colores, 
          strokewidth = 0
         );

    # --- PREPARAR DATOS DE LOS BORDES ---
    puntos_segmentos = Point2f[];
    for e in Spring_Array
        push!(puntos_segmentos, Point2f(e.S1[1] + MoveVector[1], e.S1[2] + MoveVector[2])); #Punto Inicio
        push!(puntos_segmentos, Point2f(e.S2[1] + MoveVector[1], e.S2[2] + MoveVector[2])); #Punto Fin
    end

    linesegments!(
                  Fila_Vecindades, puntos_segmentos,
                  linewidth = 1,
                  color = :black
                 );
end

fig

## Construcción de la vecindad local $N = 15$ alrededor del origen

In [ ]:
NSides = 15;                #Simetría rotacional del sistema cuasiperiódico
Areas_Values = [0.207912, 0.406737, 0.587785, 0.743145, 0.866025, 0.951057, 0.994522]; #Arreglo con las áreas, de menor a mayor, de las posibles teselas
Rho = 1.2685820147323164;   #Densidad de sitios en una vecindad cuasiperiódica (Decoración vértices)
APoint = [0, 0];            #Punto arbitrario en el que se generan las vecindades
Error_Margin = 3;           #Margen de error asociado a los enteros asociados a cada vector estrella del GDM

#Definición de Colores Base
raw_colors = [
              [255, 0, 89]/255,     #Color de la primer tesela
              [255, 140, 0]/255,    #Color de la segunda tesela
              [180, 230, 0]/255,    #Color de la tercer tesela
              [15, 255, 219]/255,   #Color de la cuarta tesela
              [10, 210, 255]/255,   #Color de la quinta tesela
              [41, 98, 255]/255,    #Color de la sexta tesela
              [149, 0, 255]/255     #Color de la septima tesela
             ];

FactNorm = 2*sqrt(π*Rho); #Factor de normalización para las longitudes, genera densidades constantes

Lambda = AproxLambda(NSides);       #Valor del aproximante estadístico (Tras la normalización de Torquato)
RadioVecindad = Lambda/FactNorm;    #Valor del radio de la vecindad circular para nuestro aproximante estadístico (Valor previo a la normalización de Torquato)

Star_Vectors = [[BigFloat(1), 0]];  #Arreglo que contendrá los vectores estrella
for i in 1:(NSides-1)
    push!(Star_Vectors, [cos((2*i)*pi/NSides), sin((2*i)*pi/NSides)]); #Vértices del polígono regular con NSides
end
Alphas_Array = fill(1e-3, NSides);  #Arreglo con las constantes alfas del GDM
Average_Distance_Stripes = fill(NSides/2, NSides); #Arreglo con la separación promedio entre las franjas cuasiperiódicas   

#Construimos los vértices de las teselas del arreglo cuasiperiódico alrededor de APoint
X_Coord_Site, Y_Coord_Site, Lattice_Sites = quasiperiodic_Neighbourhood(NSides, Error_Margin, RadioVecindad, APoint; Alpha_Value = Alphas_Array[1]); #Generamos la vecindad circular
X_Coord_Site = X_Coord_Site .- APoint[1]; #Recorremos las coordenadas de las teselas alrededor del origen
Y_Coord_Site = Y_Coord_Site .- APoint[2]; #Recorremos las coordenadas de las teselas alrededor del origen

Spring_Array = []; #Arreglo que contendrá las conexiones entre vértices
for i in 1:4:length(X_Coord_Site)
    #Obtenemos las coordenadas de cada tesela en un formato de polígono cerrado (V1-> V2-> V3-> V4-> V1)
    XX = [X_Coord_Site[i], X_Coord_Site[i+1], X_Coord_Site[i+2], X_Coord_Site[i+3], X_Coord_Site[i]]
    YY = [Y_Coord_Site[i], Y_Coord_Site[i+1], Y_Coord_Site[i+2], Y_Coord_Site[i+3], Y_Coord_Site[i]]

    #Conectamos a pares los vértices de las teselas para generar los 4 lados de la misma (V1-V2, V2-V3, V3-V4, V4-V1)
    for j in 1:(length(XX) - 1)
        Tile_Side = Spring([XX[j], YY[j]], [XX[j+1], YY[j+1]]) #Definimos el lado de la tesela que conecta a dos vértices subsecuentes
        Add_Side = true; #Una llave que determina si se guarda la conexión al ser única o se descarta por ser repetida

        for e in Spring_Array #Iteramos sobre todas las conexiones previas
            if is_Equal(e, Tile_Side) #Verificamos si el lado recién creado es una nueva conexión
                Add_Side = false; #Si a conexión recien creada es igual a una previa, se descarta su adición y se rompe el ciclo
                break
            end
        end

        if Add_Side == true
            push!(Spring_Array, Tile_Side) #Si tras revisar con todos las conexiones previas no se encuentra un duplicado, entonces se añade la nueva conexión
        end
    end
end

palette_rgb = [RGB(c[1], c[2], c[3]) for c in raw_colors]; #Convertimos a objetos RGB para manipularlos fácilmente

# --- PREPARAR DATOS DE LAS TESELAS (POLÍGONOS) ---
#Listas vacías para acumular los polígonos y sus colores
lista_poligonos = Vector{Vector{Point2f}}();
lista_colores = Vector{RGBA}();

#Iteramos sobre el índice del primer vértice que conforma cada tesela
for i in 1:4:length(X_Coord_Site)
    #Extraemos coordenadas (XTile, YTile)
    xs = [X_Coord_Site[k] for k in i:(i+3)];
    ys = [Y_Coord_Site[k] for k in i:(i+3)];
        
    #Calculamos el Área
    XTile_calc = [xs; xs[1]];
    YTile_calc = [ys; ys[1]];
    A = round(Float64(Area(XTile_calc, YTile_calc)), digits = 6);
        
    #Buscar índice del color
    Index = findfirst(x -> x == A, Areas_Values);
        
    #Determinar Alpha y Color final
    alpha_val = (Index == 1) ? 1.0 : Valor_Alpha_Teselas;
    base_c = palette_rgb[Index];
        
    #Guardamos el polígono (como lista de puntos Point2f) y su color RGBA
    push!(lista_poligonos, Point2f.(xs, ys));
    push!(lista_colores, RGBA(base_c.r, base_c.g, base_c.b, alpha_val));
end

# --- GRAFICAR TODAS LAS TESELAS DE UNA VEZ ---
poly!(
      Local_Origin_N15, lista_poligonos, 
      color = lista_colores, 
      strokewidth = 0
     );

# --- PREPARAR DATOS DE LOS BORDES ---
puntos_segmentos = Point2f[];
for e in Spring_Array
    push!(puntos_segmentos, Point2f(e.S1[1], e.S1[2])); #Punto Inicio
    push!(puntos_segmentos, Point2f(e.S2[1], e.S2[2])); #Punto Fin
end

linesegments!(
              Local_Origin_N15, puntos_segmentos,
              linewidth = 1,
              color = :black
             );

fig

## Aproximante clásico teselas más pequeñas y vecindad local

Teselas más pequeñas del aproximante clásico y el marco azul que delimita el área

In [ ]:
NSides = 11;                #Simetría rotacional del sistema cuasiperiódico
Areas_Values = [0.281733, 0.540641, 0.75575, 0.909632, 0.989821]; #Arreglo con las áreas, de menor a mayor, de las posibles teselas
Rho = 1.2645739922427748;   #Densidad de sitios en una vecindad cuasiperiódica (Decoración vértices)
FactNorm = 2*sqrt(π*Rho);   #Factor de normalización para las longitudes, genera densidades constantes
AreaNumber = 1;             #Tamaño del área a considerar para graficar

#Dirección donde se encuentra el archivo .csv con las coordenadas de las teselas más pequeñas del aproximante clásico del sistema cuasiperiódico a considerar
PathCoord = "Quasiperiodic-Tiles/Global Structural Studies/Data/Fig1_SmallestTilesCoord/A1/"

#Las teselas corresponden a una vecindad cuadrada generada alrededor de un punto aleatorio pero desplazado al origen posteriormente
APoint = [710710.3918523701, -517876.9206462959]; #Centro de la vecindad generada inicialmente
LadoVecindad = 530; #Tamaño de la vecindad cuadrada inscrita en la vecindad circular de la que proceden las teselas (corresponde al valor de AproxClasico(NSides))
CoordTeselas = readdlm(PathCoord * "Coordenadas_Vertices_Teselas_N$(NSides)_LAC$(LadoVecindad)_Area$(AreaNumber).csv");

############################################# BACKGROUND TESELAS MÁS PEQUEÑAS ###############################################
#Graficamos las teselas más pequeñas. Por seguridad de memoria no se muestra en el notebook, se guarda directamente en un PDF
XCoord = CoordTeselas[:,1]; #Coordenadas X de las teselas, en grupos de 5 (Formato de polígono cerrado V1->V2->V3->V4->V1)
YCoord = CoordTeselas[:,2]; #Coordenadas Y de las teselas, en grupos de 5 (Formato de polígono cerrado V1->V2->V3->V4->V1)

# --- PREPARAR DATOS DE LAS TESELAS (POLÍGONOS) ---
#Listas vacías para acumular los polígonos y sus colores
lista_poligonos = Vector{Vector{Point2f}}();

#Iteramos sobre el índice del primer vértice que conforma cada tesela
for i in 1:5:length(XCoord)
    #Extraemos coordenadas (XTile, YTile)
    xs = [XCoord[k] for k in i:(i+3)];
    ys = [YCoord[k] for k in i:(i+3)];
    
    #Calculamos el Área
    XTile_calc = [xs; xs[1]];
    YTile_calc = [ys; ys[1]];
    A = round(Float64(Area(XTile_calc, YTile_calc)), digits = 6);
    
    #Buscar índice del color
    Index = findfirst(x -> x == A, Areas_Values);
    
    #Guardamos el polígono (como lista de puntos Point2f) y su color RGBA
    push!(lista_poligonos, Point2f.(xs, ys));
end

# --- GRAFICAR TODAS LAS TESELAS DE UNA VEZ ---
poly!(
      Approx_Clasico, lista_poligonos, 
      color = :black, 
      strokewidth = 0.1
     )

#Graficamos el marco azul
marco_x = [-LadoVecindad/2, -LadoVecindad/2, LadoVecindad/2, LadoVecindad/2, -LadoVecindad/2];
marco_y = [-LadoVecindad/2, LadoVecindad/2, LadoVecindad/2, -LadoVecindad/2, -LadoVecindad/2];

lines!(
       Approx_Clasico, marco_x, marco_y, 
       color = RGB(38/255, 139/255, 253/255), 
       linewidth = 5
      );

Vecindad local en la parte superior del aproximante clásico

In [ ]:
Lambda = AproxLambda(NSides);       #Valor del aproximante estadístico (Tras la normalización de Torquato)
RadioVecindad = Lambda/FactNorm;    #Valor del radio de la vecindad circular para nuestro aproximante estadístico (Valor previo a la normalización de Torquato)

Star_Vectors = [[BigFloat(1),0]];   #Arreglo que contendrá los vectores estrella
for i in 1:(NSides-1)
    push!(Star_Vectors, [cos((2*i)*pi/NSides), sin((2*i)*pi/NSides)]); #Vértices del polígono regular con NSides
end
Alphas_Array = fill(0.0, NSides);   #Arreglo con las constantes alfas del GDM
Average_Distance_Stripes = fill(NSides/2, NSides); #Arreglo con la separación promedio entre las franjas cuasiperiódicas

Error_Margin = 3;       #Margen de error asociado a los enteros asociados a cada vector estrella del GDM
MargenIzquierdo = 190;  #Margen que se proporciona para que el extremo izquierdo de la vecindad circular no se encime al borde izquierdo de la vecindad cuadrada
MargenSuperior = 10;    #Margen que se proporciona para que el extremo superior de la vecindad circular no se encime al borde superior de la vecindad cuadrada
APoint2 = [
           APoint[1] - LadoVecindad/2 + RadioVecindad + MargenIzquierdo,
           APoint[2] + LadoVecindad/2 - RadioVecindad - MargenSuperior
          ];

#Construimos los vértices de las teselas del arreglo cuasiperiódico alrededor de APoint2
X_Coord_Site, Y_Coord_Site, Lattice_Sites = quasiperiodic_Neighbourhood(NSides, Error_Margin, RadioVecindad, APoint2); #Generamos la vecindad circular
X_Coord_Site = X_Coord_Site .- APoint[1]; #Recorremos las coordenadas de las teselas alrededor del origen
Y_Coord_Site = Y_Coord_Site .- APoint[2]; #Recorremos las coordenadas de las teselas alrededor del origen

Spring_Array = []; #Arreglo que contendrá las conexiones entre vértices
for i in 1:4:length(X_Coord_Site)
    #Obtenemos las coordenadas de cada tesela en un formato de polígono cerrado (V1-> V2-> V3-> V4-> V1)
    XX = [X_Coord_Site[i], X_Coord_Site[i+1], X_Coord_Site[i+2], X_Coord_Site[i+3], X_Coord_Site[i]];
    YY = [Y_Coord_Site[i], Y_Coord_Site[i+1], Y_Coord_Site[i+2], Y_Coord_Site[i+3], Y_Coord_Site[i]];

    #Conectamos a pares los vértices de las teselas para generar los 4 lados de la misma (V1-V2, V2-V3, V3-V4, V4-V1)
    for j in 1:(length(XX) - 1)
        Tile_Side = Spring([XX[j], YY[j]], [XX[j+1], YY[j+1]]) #Definimos el lado de la tesela que conecta a dos vértices subsecuentes
        Add_Side = true; #Una llave que determina si se guarda la conexión al ser única o se descarta por ser repetida

        for e in Spring_Array #Iteramos sobre todas las conexiones previas
            if is_Equal(e, Tile_Side) #Verificamos si el lado recién creado es una nueva conexión
                Add_Side = false; #Si a conexión recien creada es igual a una previa, se descarta su adición y se rompe el ciclo
                break
            end
        end

        if Add_Side == true
            push!(Spring_Array, Tile_Side); #Si tras revisar con todos las conexiones previas no se encuentra un duplicado, entonces se añade la nueva conexión
        end
    end
end

################################################ VECINDAD LOCAL N = 11 ###############################################
#Definición de Colores Base
raw_colors = [
              [214, 40, 40]/255,    #Color de la primer tesela
              [47, 72, 88]/255,     #Color de la segunda tesela
              [124, 181, 24]/255,   #Color de la tercer tesela
              [134, 187, 216]/255,  #Color de la cuarta tesela
              [242, 100, 25]/255    #Color de la quinta tesela
             ];
palette_rgb = [RGB(c[1], c[2], c[3]) for c in raw_colors]; #Convertimos a objetos RGB para manipularlos fácilmente

#Preparación de contenedores
lista_poligonos = [];
lista_colores = RGB[];

for i in 1:4:length(X_Coord_Site)
    #Extraemos los 4 vértices únicos para Makie
    x_verts = X_Coord_Site[i:(i+3)];
    y_verts = Y_Coord_Site[i:(i+3)];
    
    #Extraemos los 5 vértices (repetido el primero) para el cálculo del área
    XTile_Calc = [x_verts; x_verts[1]];
    YTile_Calc = [y_verts; y_verts[1]];
    
    A = round(Float64(Area(XTile_Calc, YTile_Calc)), digits = 6);
    Index = findfirst(x -> x == A, Areas_Values);
    ColorTesela = palette_rgb[Index];

    #Creamos el polígono para Makie y lo guardamos
    pts = Point2f.(x_verts, y_verts);
    push!(lista_poligonos, pts);
    
    #Guardamos el color correspondiente
    push!(lista_colores, ColorTesela);
end

poly!(Approx_Clasico, lista_poligonos, color = lista_colores, strokewidth = 0);

#Graficamos las líneas verticales que señalan la distancia 2*λ_N
vlines!(
        Approx_Clasico, [APoint2[1] - APoint[1] - RadioVecindad, APoint2[1] - APoint[1] + RadioVecindad],
        color = RGB(255/255, 215/255, 0/255),
        linewidth = 3,
        alpha = 0.4
       )

fig

## Agregamos las gráficas correspondientes a la hiperuniformidad en 2D

In [ ]:
PathNR = "Quasiperiodic-Tiles/Global Structural Studies/Data/Fig1_SigmaSquare/"; #Ruta de acceso a la carpeta con los datos de la envolvente superior del ConcaveHull de la gR

#Información de la primera gráfica de Varianza Sigma^2
NSides1 = 11;        #Simetría rotacional de la primera gráfica del Sigma^2
RandSteps1 = 0;      #Número de pasos de randomización del algoritmo empleados por el primer Sigma^2
Radius1 = 500;       #Radio máximo analizado para el primer Sigma^2 (Antes Torquato)

#Matriz de Nx1 con los valores Y de la varianza. NR[:,1] regresa las coordenadas del eje Y
NR1 = readdlm(PathNR * "Torquato_NR_N$(NSides1)_Alfa0P0_R$(Radius1)_Step1e5_SigmaCuadrada.csv");

#Información de la segunda gráfica de Varianza Sigma^2
NSides2 = 19;        #Simetría rotacional de la segunda gráfica del Sigma^2
RandSteps2 = 0;      #Número de pasos de randomización del algoritmo empleados por el segundo Sigma^2
Radius2 = 500;       #Radio máximo analizado para el segundo Sigma^2 (Antes Torquato)

#Matriz de Nx1 con los valores Y de la varianza. NR[:,1] regresa las coordenadas del eje Y
NR2 = readdlm(PathNR * "Torquato_NR_N$(NSides2)_Alfa0P0_R$(Radius2)_Step1e5_SigmaCuadrada.csv");

#Información de la tercera gráfica de Varianza Sigma^2
NSides3 = 25;        #Simetría rotacional de la tercera gráfica del Sigma^2
RandSteps3 = 0;      #Número de pasos de randomización del algoritmo empleados por el tercer Sigma^2
Radius3 = 500;       #Radio máximo analizado para el tercer Sigma^2 (Antes Torquato)

#Matriz de Nx1 con los valores Y de la varianza. NR[:,1] regresa las coordenadas del eje Y
NR3 = readdlm(PathNR * "Torquato_NR_N$(NSides3)_Alfa0P0_R$(Radius3)_Step1e5_SigmaCuadrada.csv");

# --- PRIMERA GRÁFICA ---
Rho1 = 1.2645739922427748;
Step1 = 100000;
R1 = range(Radius1 / Step1, stop = Radius1, length = Step1);
R1 = (2*sqrt(π*Rho1)) .* R1;

UpValue1 = 0; #Valor de desplazamiento vertical para la primera gráfica
lines!(
       Sigma2_2D, R1, UpValue1 .+ (NR1[:,1] ./ R1),
       color = :blue,
       alpha = 1,
       linewidth = 1.5
      );

# --- SEGUNDA GRÁFICA ---
Rho2 = 1.2703373895977548;
Step2 = 100000;
R2 = range(Radius2 / Step2, stop = Radius2, length = Step2);
R2 = (2*sqrt(π*Rho2)) .* R2;

UpValue2 = 0.5; #Valor de desplazamiento vertical para la segunda gráfica
lines!(
       Sigma2_2D, R2, UpValue2 .+ (NR2[:,1] ./ R2),
       color = :orange,
       alpha = 1
      );

# --- TERCERA GRÁFICA ---
Rho3 = 1.271563821555834;
Step3 = 100000;
R3 = range(Radius3 / Step3, stop = Radius3, length = Step3);
R3 = (2*sqrt(π*Rho3)) .* R3;

UpValue3 = 1.2; #Valor de desplazamiento vertical para la segunda gráfica
lines!(
       Sigma2_2D, R3, UpValue3 .+ (NR3[:,1] ./ R3),
       color = :green,
       alpha = 1
      );

# --- Múltiplos de Lambda 1 ---
Lambda1 = AproxLambda(NSides1);
for i in 1:12  
    lines!(
           Sigma2_2D, [i*Lambda1, i*Lambda1], [UpValue1 + 0.0, UpValue1 + 0.55],
           linestyle = :dot,
           linewidth = 3,
           color = :darkblue
          );
end

# --- Múltiplos de Lambda 2 ---
Lambda2 = AproxLambda(NSides2);
for i in 1:4    
    lines!(
           Sigma2_2D, [i*Lambda2, i*Lambda2], [UpValue2 + 0.15, UpValue2 + 0.9],
           linestyle = :dot,
           linewidth = 3,
           color = :darkred
          );
end

# --- Múltiplos de Lambda 3 ---
Lambda3 = AproxLambda(NSides3);
for i in 1:2
    lines!(
           Sigma2_2D, [i*Lambda3, i*Lambda3], [UpValue3 + 0.2, UpValue3 + 1.2],
           linestyle = :dot,
           linewidth = 3,
           color = :darkgreen
          );
end

# --- SEÑALIZACIÓN DE LA ESCALA DE LONGITUD KAPPA ---
#Parámetros para N = 23
Rho = 1.271563821555834;           #Densidad de sitios en una vecindad cuasiperiódica (Decoración vértices)
FactNorm = 2*sqrt(π*Rho);          #Factor de normalización para las longitudes, genera densidades constantes
κ_25 = (NSides3 / 4) * FactNorm;   #Valor de la escala de longitud Kappa
Index_Kappa = findfirst(x -> x >= κ_25, R3);
scatter!(
         Sigma2_2D, [κ_25], [(NR3[Index_Kappa] ./ R3[Index_Kappa]) + UpValue3],
         color = :red,
         marker = :diamond,
         alpha = 0.6
        );

#Parámetros para N = 19
Rho = 1.2703373895977548;          #Densidad de sitios en una vecindad cuasiperiódica (Decoración vértices)
FactNorm = 2*sqrt(π*Rho);          #Factor de normalización para las longitudes, genera densidades constantes
κ_19 = (NSides2 / 4) * FactNorm;   #Valor de la escala de longitud Kappa
Index_Kappa = findfirst(x -> x >= κ_19, R2);
scatter!(
         Sigma2_2D, [κ_19], [(NR2[Index_Kappa] ./ R2[Index_Kappa]) + UpValue2],
         color = :red,
         marker = :diamond,
         alpha = 0.6
        );

#Parámetros para N = 11
Rho = 1.2645739922427748;          #Densidad de sitios en una vecindad cuasiperiódica (Decoración vértices)
FactNorm = 2*sqrt(π*Rho);          #Factor de normalización para las longitudes, genera densidades constantes
κ_11 = (NSides1 / 4) * FactNorm;   #Valor de la escala de longitud Kappa
Index_Kappa = findfirst(x -> x >= κ_11, R1);
scatter!(
         Sigma2_2D, [κ_11], [(NR1[Index_Kappa] ./ R1[Index_Kappa]) + UpValue1],
         color = :red,
         marker = :diamond,
         alpha = 0.6
        );   

fig

## Agregamos las gráficas correspondientes a $g(R)$ en 2D

In [ ]:
PathNR = "Quasiperiodic-Tiles/Global Structural Studies/Data/Fig1_ConcaveHull_gR/"; #Ruta de acceso a la carpeta con los datos de la envolvente superior del ConcaveHull de la gR

#Información de la primera Concave Hull a graficar
NSides1 = 11;        #Simetría rotacional de la primera gráfica del Concave Hull
RandSteps1 = 0;      #Número de pasos de randomización del algoritmo empleados por el primer Concave Hull
RTorquato1 = 479;    #Radio máximo analizado para el primer Concave Hull (Tras Torquato)
PCH1 = 25;           #Número de vecinos considerados durante la creación del Concave Hull
m1 = 0;              #Número de vecinos considerados durante la "suavización" de la gráfica de la envolvente con una media móvil

#Matriz de Nx2 con los valores X y Y de la envolvente superior. CH[:,1] regresa las coordenadas del eje X y CH[:,2] regresa las coordenadas del eje Y
CH1 = readdlm(PathCH * "gR_ConcaveHull_VerticesDeco_SpecialData_N$(NSides1)_rand$(RandSteps1)_Alfa0P0_RTorquato$(RTorquato1)_Step0P001_PCH$(PCH1)_m$(m1).csv");

#Información de la segunda Concave Hull a graficar
NSides2 = 19;        #Simetría rotacional de la primera gráfica del Concave Hull
RandSteps2 = 0;      #Número de pasos de randomización del algoritmo empleados por el primer Concave Hull
RTorquato2 = 479;    #Radio máximo analizado para el primer Concave Hull (Tras Torquato)
PCH2 = 50;           #Número de vecinos considerados durante la creación del Concave Hull
m2 = 0;              #Número de vecinos considerados durante la "suavización" de la gráfica de la envolvente con una media móvil

#Matriz de Nx2 con los valores X y Y de la envolvente superior. CH[:,1] regresa las coordenadas del eje X y CH[:,2] regresa las coordenadas del eje Y
CH2 = readdlm(PathCH * "gR_ConcaveHull_VerticesDeco_SpecialData_N$(NSides2)_rand$(RandSteps2)_Alfa0P0_RTorquato$(RTorquato2)_Step0P001_PCH$(PCH2)_m$(m2).csv");

#Información de la tercera Concave Hull a graficar
NSides3 = 25;        #Simetría rotacional de la primera gráfica del Concave Hull
RandSteps3 = 0;      #Número de pasos de randomización del algoritmo empleados por el primer Concave Hull
RTorquato3 = 479;    #Radio máximo analizado para el primer Concave Hull (Tras Torquato)
PCH3 = 50;           #Número de vecinos considerados durante la creación del Concave Hull
m3 = 0;              #Número de vecinos considerados durante la "suavización" de la gráfica de la envolvente con una media móvil

#Matriz de Nx2 con los valores X y Y de la envolvente superior. CH[:,1] regresa las coordenadas del eje X y CH[:,2] regresa las coordenadas del eje Y
CH3 = readdlm(PathCH * "gR_ConcaveHull_VerticesDeco_SpecialData_N$(NSides3)_rand$(RandSteps3)_Alfa0P0_RTorquato$(RTorquato3)_Step0P001_PCH$(PCH3)_m$(m3).csv");

# --- PRIMERA ENVOLVENTE ---
UpValue1 = 1;
lines!(
       gR_2D, CH1[:,1], UpValue1 .* CH1[:,2],
       color = :blue,
       linewidth = 1.5,
       alpha = 1
      );

# --- SEGUNDA ENVOLVENTE ---
UpValue2 = 75;
lines!(
       gR_2D, CH2[:,1], UpValue2 .* CH2[:,2],
       color = :orange,
       linewidth = 1.5,
       alpha = 1
      );

# --- TERCERA ENVOLVENTE ---
UpValue3 = 1800;
lines!(
       gR_2D, CH3[:,1], UpValue3 .* CH3[:,2],
       color = :green,
       linewidth = 1.5,
       alpha = 1
      );

# --- Múltiplos de 2*Lambda ---
Lambda1 = AproxLambda(NSides1)
for i in 2:2:10    
    lines!(
           gR_2D, [i*Lambda1, i*Lambda1], [UpValue1, UpValue1*(1 + 100)],
           linestyle = :dot,
           linewidth = 3,
           color = :darkblue
          );
end

# --- Múltiplos de 2*Lambda ---
Lambda2 = AproxLambda(NSides2)
for i in 2:2:4    
    lines!(
           gR_2D, [i*Lambda2, i*Lambda2], [UpValue2*(1 + 0.5), UpValue2*(1 + 20)],
           linestyle = :dot,
           linewidth = 3,
           color = :darkred
          );
end

# --- Múltiplos de 2*Lambda ---
Lambda3 = AproxLambda(NSides3)
for i in 2:2:2    
    lines!(
           gR_2D, [i*Lambda3, i*Lambda3], [UpValue3, UpValue3*(1 + 15)],
           linestyle = :dot,
           linewidth = 3,
           color = :darkgreen
          );
end

fig

## Agregamos las gráficas correspondientes a $λ_{N}$ como función de $N$ en los casos 1D y 2D

In [ ]:
# --- Valor teórico del aproximante clásico 2D ---
AproxClasico(x) = 0.3318*exp(0.6705*x);
#Discretización del dominio
x_vals = range(5, stop = 22.5, length = 2000)
y_vals = AproxClasico.(x_vals)
lines!(
       λ_N, x_vals, y_vals,
       linestyle = :dot,
       label = L"y = 0.33 \ \exp(0.67 x) \ (2D)"
      );

# --- Valor teórico del caso 2D ---
#Discretización del dominio
x_vals = range(5, stop = 51, length = 2000)
y_vals = AproxLambda.(x_vals)
lines!(
       λ_N, x_vals, y_vals,
       label = L"y = \frac{2 \pi}{1 - \cos(2π/N)} \ (2D)"
      );
# --- Valor ajustado por Fourier del caso 2D ---
#Nota: Estos valores se obtuvieron haciendo un ajuste de Fourier a los datos de las varianzas para cada simetría rotacional por separado.
#Nota2: El caso N = 5 es demasiado ruidoso, no hay una manera clara de determinar el valor de lambda. El caso N = 7 corresponde al segundo pico, el primer pico da los valores de las oscilaciones intermedias.
y_vals_Fourier = [1.879714, 15.49654, 25.84154, 38.33043, 53.92222, 71.29781, 90.77957, 117.5129, 142.7238, 166.537, 199.8683, 222.0968, 285.5745, 333.1905]
y_vals_Fourier = [15.49654, 25.84154, 38.33043, 53.92222, 71.29781, 90.77957, 117.5129, 142.7238, 166.537, 199.8683, 222.0968, 285.5745, 333.1905]
scatter!(
         λ_N, 7:2:31, y_vals_Fourier,
         label = L"Experimental \ Data",
         markersize = 5,
         color = :red
        );

# --- Valor teórico del caso 1D ---
#Discretización del dominio
x_vals = range(5, stop = 51, length = 2000)
y_vals = x_vals ./ 2
lines!(
       λ_N, x_vals, y_vals,
       label = L"y = N/2 \ (1D)"
      );
# --- Valor ajustado por Fourier del caso 1D ---
#Nota: Estos valores se obtuvieron haciendo un ajuste de Fourier.
y_vals_Fourier = [
                  0.72,
                  3.50194552529182,
                  4.22535211267605,
                  4.62724935732647,
                  6.49819494584837,
                  7.3469387755102,
                  7.9295154185022,
                  9.375,
                  10.4046242774566,
                  11.3924050632911,
                  12.1452702702702,
                  12.6760563380281,
                  14.4475806451612,
                  14.7540983606557,
                  15.7894736842105,
                  17.4757281553398,
                  16.822429906542,
                  18.9473684210526,
                  20.4545454545454,
                  21.4285714285714,
                  21.0764705882352,
                  23.3766233766233,
                  24,
                  25.3521126760563
                 ]
scatter!(
         λ_N, 5:2:51, y_vals_Fourier,
         label = L"Experimental \ Data",
         markersize = 5,
         color = :magenta
        );

fig